# Chapter 11: Social Network Analysis

**Foundations of Data Science**

## In this lesson

1. When the structure is the data: nodes, edges, weights
2. From a contact log to a graph
3. Describing a network: size, density, degree
4. Who matters: four kinds of centrality and what each one means
5. Who mixes with whom: role-level structure and assortativity
6. Communities: finding groups from structure alone
7. Spreading on a network, and where one intervention does the most good
8. Time, and what aggregation throws away
9. Pitfalls

## The one idea

**In a network, your risk depends on who you're connected to, not only on who you are.** Every method before this chapter treated rows as independent people. Here the relationships between rows are the data, and the questions change: not "what predicts this patient's outcome" but "if something starts here, where does it go, and where would one intervention stop the most of it?"

The setting: a short-stay ward in a Lyon hospital, December 2010. Every participant (29 patients, 27 nurses, 11 doctors, 8 administrative staff) wore a proximity sensor for four days; every face-to-face contact was logged at 20-second resolution (Vanhems et al., PLoS ONE 2013, via the SocioPatterns collaboration). **The infection-control question: if the ward could isolate or vaccinate a handful of people to slow a hospital-acquired infection, who, and why?**

## Setup

<!-- nb-links -->
[Download this notebook](chapter_11_networks.ipynb?download=1){download="chapter_11_networks.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/chapter_11/chapter_11_networks.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/chapter_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("chapter_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
if not Path("data/raw/hospital_contacts.parquet").exists():
    !python data_gen.py fetch --name hospital_contacts

print("Working directory:", os.getcwd())

## 1. From a contact log to a graph

The raw data is one row per 20-second interval per pair in contact: 32,424 rows. A **graph** collapses that into **nodes** (people) and **edges** (pairs who were ever in contact), with an edge **weight** for how much: here, total contact time in minutes. Each node carries a **role** attribute. This aggregation is the first modeling decision of the chapter, and section 8 comes back to what it loses.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import warnings
warnings.filterwarnings("ignore")

contacts = pd.read_parquet("data/raw/hospital_contacts.parquet")
print(contacts.shape, " time span:", round((contacts["t"].max() - contacts["t"].min()) / 86400, 1), "days")

# Roles per person
roles = pd.concat([contacts[["i", "role_i"]].set_axis(["id", "role"], axis=1),
                   contacts[["j", "role_j"]].set_axis(["id", "role"], axis=1)]).drop_duplicates().set_index("id")["role"]
print(roles.value_counts().to_dict())

# Aggregate: one edge per unordered pair, weight = total contact minutes (each row is 20 seconds)
pair = contacts.assign(a=np.minimum(contacts["i"], contacts["j"]), b=np.maximum(contacts["i"], contacts["j"]))
edges = pair.groupby(["a", "b"]).size().mul(20 / 60).rename("minutes").reset_index()

G = nx.Graph()
G.add_nodes_from((n, {"role": r}) for n, r in roles.items())
G.add_weighted_edges_from(edges[["a", "b", "minutes"]].itertuples(index=False), weight="minutes")
print("nodes:", G.number_of_nodes(), " edges:", G.number_of_edges(), " total contact hours:", round(edges["minutes"].sum() / 60, 1))

## 2. Describing a network

Three numbers to start. **Density**: the share of possible pairs that are connected. **Degree**: how many distinct people each person touched. **Strength**: weighted degree, total minutes of contact. Degree counts breadth; strength counts exposure. They disagree more often than you'd expect.

In [ ]:
deg = pd.Series(dict(G.degree()), name="degree")
strength = pd.Series(dict(G.degree(weight="minutes")), name="minutes")
node_df = pd.concat([roles, deg, strength], axis=1)

print("density:", round(nx.density(G), 3), "  (share of the", G.number_of_nodes() * (G.number_of_nodes() - 1) // 2, "possible pairs that ever met)")
print("\nby role:")
node_df.groupby("role").agg(n=("degree", "size"), mean_degree=("degree", "mean"), mean_minutes=("minutes", "mean"), max_minutes=("minutes", "max")).round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
node_df["degree"].plot.hist(bins=20, ax=axes[0], color="C0", alpha=0.8); axes[0].set_xlabel("distinct contacts (degree)"); axes[0].set_title("Degree")
node_df["minutes"].plot.hist(bins=20, ax=axes[1], color="C1", alpha=0.8); axes[1].set_xlabel("total contact minutes (strength)"); axes[1].set_title("Strength")
plt.show()

Nurses have the most contacts and the most contact time; patients the fewest. Both distributions are right-skewed: a few people account for a large share of all contact, which is the single most important fact for spreading (section 7).

## 3. Drawing it

A network picture is for pattern, not precision. Nodes colored by role, placed by a force-directed layout (connected nodes pull together), edges thickened by contact time. Look for: does the ward cluster by role? Who sits between clusters?

In [ ]:
role_colors = {"PAT": "#999999", "NUR": "#1f77b4", "MED": "#2ca02c", "ADM": "#ff7f0e"}
pos = nx.spring_layout(G, weight="minutes", seed=4, k=0.35)

fig, ax = plt.subplots(figsize=(11, 9))
w = np.array([G[u][v]["minutes"] for u, v in G.edges()])
nx.draw_networkx_edges(G, pos, width=0.2 + 2.5 * w / w.max(), alpha=0.25, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=[role_colors[G.nodes[n]["role"]] for n in G], node_size=60 + 6 * deg[list(G)].values, ax=ax, linewidths=0.5, edgecolors="black")
for r, c in role_colors.items():
    ax.scatter([], [], c=c, label=r, s=60, edgecolors="black", linewidths=0.5)
ax.legend(title="role", loc="upper left"); ax.set_axis_off()
ax.set_title("Hospital ward contact network: node size = degree, edge width = contact minutes")
plt.show()

## 4. Who matters: four centralities

"Important" has several meanings in a network, and each has a measure:

| Centrality | Question it answers | Computed as |
|---|---|---|
| **Degree** | who touches the most people? | count of neighbors |
| **Strength** | who has the most exposure time? | sum of edge weights |
| **Betweenness** | who sits on the paths between others? | share of shortest paths through the node |
| **Eigenvector** | who is connected to well-connected people? | a node's score is the weighted sum of its neighbors' scores (solved as an eigenvector, hence the name) |

For infection control, betweenness is the one to watch: a person with modest contact time who is the *only* link between patients and the doctors' office is a bridge, and bridges are where spreading crosses groups.

In [ ]:
cent = node_df.copy()
cent["betweenness"] = pd.Series(nx.betweenness_centrality(G, weight=None))          # path-based: unweighted is the usual choice
cent["eigenvector"] = pd.Series(nx.eigenvector_centrality(G, weight="minutes", max_iter=1000))
for c in ["degree", "minutes", "betweenness", "eigenvector"]:
    cent[f"rank_{c}"] = cent[c].rank(ascending=False).astype(int)

print("top 8 by betweenness:")
print(cent.sort_values("betweenness", ascending=False).head(8)[["role", "degree", "minutes", "betweenness", "eigenvector"]].round(3).to_string())
print("\nrank correlation between measures (Spearman):")
print(cent[["degree", "minutes", "betweenness", "eigenvector"]].corr(method="spearman").round(2))

Read the top-8 table against the rank correlations. The measures agree broadly (Spearman 0.82 to 0.95, nurses fill most of every list) but not at the top. The single highest-betweenness person is an **administrative staffer**: the most distinct contacts in the ward (61 of 74 possible) but middling contact time. That is what a bridge looks like: brief contact with everyone, long contact with no one. By minutes they wouldn't make the top ten. If the ward can vaccinate five staff, "the five with the most contact time" and "the five on the most paths" are different lists, and section 7 tests which list works better.

## 5. Who mixes with whom

Collapse the graph to roles: a 4-by-4 matrix of contact minutes between role pairs. Then **assortativity**: do people mostly contact their own role (positive), other roles (negative), or neither (zero)? For a hospital this is the mechanism of transmission between groups, and it's also a sanity check on the data (nurses should contact patients a lot; admin staff should not).

In [ ]:
mix = pd.crosstab(contacts["role_i"], contacts["role_j"], values=contacts["t"], aggfunc="size").fillna(0) * 20 / 60
mix = (mix + mix.T) / 2                                   # make symmetric: contacts are undirected
mix = mix.loc[["PAT", "NUR", "MED", "ADM"], ["PAT", "NUR", "MED", "ADM"]].round(0)

import seaborn as sns
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(mix, annot=True, fmt=".0f", cmap="Blues", ax=ax, cbar_kws={"label": "contact minutes"})
ax.set_title("Contact minutes between roles (symmetric)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.show()

print("role assortativity:", round(nx.attribute_assortativity_coefficient(G, "role"), 3))
print("patient-to-patient minutes:", int(mix.loc["PAT", "PAT"]), "  patient-to-nurse:", int(mix.loc["PAT", "NUR"]))

Patients barely contact each other; almost all patient contact is with nurses. Nurses contact everyone. The assortativity near zero says the ward is neither clustered by role nor deliberately mixed; it's a hub-and-spoke structure with nurses at the hub. For an infection starting in a patient, the route to every other patient runs through a nurse.

## 6. Communities

Community detection finds groups of nodes that connect more among themselves than to the rest, using structure alone, no labels. **Modularity** scores a partition: how much denser within-group connection is than chance. The **Louvain** algorithm greedily maximizes it. Chapter 9's rule applies: describe the communities first, then compare them to something you held back (here, roles) with the adjusted Rand index.

In [ ]:
from sklearn.metrics import adjusted_rand_score

comms = nx.community.louvain_communities(G, weight="minutes", seed=1)
comm_of = {n: k for k, c in enumerate(comms) for n in c}
cent["community"] = pd.Series(comm_of)
print("communities:", len(comms), "  modularity:", round(nx.community.modularity(G, comms, weight="minutes"), 3))
print("\nrole composition of each community:")
print(pd.crosstab(cent["community"], cent["role"]).to_string())
print("\nagreement with roles (ARI):", round(adjusted_rand_score(cent["role"], cent["community"]), 2))

Communities mostly don't track roles. One is the entire medical staff plus the six patients they spent the most time with; the others are nurses and their patients, which is a ward's actual working structure (rooms and shifts), not its org chart. That's a finding: contact-based groups, not job titles, are the units an outbreak would move within. The low ARI against roles is the point, not a failure.

## 7. Spreading, and where one intervention helps most

A simple epidemic on the graph: start with one infected patient; each minute of contact with an infected person carries a small chance of transmission; run for a fixed time; count how many got infected. Repeat from many random starting patients to get a distribution of outbreak sizes. Then remove (isolate or vaccinate) k people chosen three ways and see which choice shrinks outbreaks most:

- **random** k people (the baseline)
- the k with the most **contact time**
- the k with the highest **betweenness**

This is a crude model (no recovery, no time structure, static contacts), but it's enough to compare *strategies*, which is what the ward asked.

In [ ]:
def outbreak_size(graph: nx.Graph, start: int, p_per_minute: float = 0.01, rng=None) -> int:
    # Susceptible-Infected spread on a weighted static graph: each edge transmits with prob 1 - (1-p)^minutes
    infected = {start}
    frontier = [start]
    while frontier:
        nxt = []
        for u in frontier:
            for v, d in graph[u].items():
                if v not in infected and rng.random() < 1 - (1 - p_per_minute) ** d["minutes"]:
                    infected.add(v); nxt.append(v)
        frontier = nxt
    return len(infected)

def simulate(graph: nx.Graph, n_runs: int = 300, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    patients = [n for n in graph if graph.nodes[n]["role"] == "PAT"]
    return np.array([outbreak_size(graph, rng.choice(patients), rng=rng) for _ in range(n_runs)])

k = 5
rng = np.random.default_rng(1)
strategies = {
    "no intervention": [],
    f"remove {k} random": list(rng.choice(list(G), size=k, replace=False)),
    f"remove top {k} by contact minutes": cent.sort_values("minutes", ascending=False).head(k).index.tolist(),
    f"remove top {k} by betweenness": cent.sort_values("betweenness", ascending=False).head(k).index.tolist(),
}
results = {}
for name, removed in strategies.items():
    H = G.copy(); H.remove_nodes_from(removed)
    sizes = simulate(H)
    results[name] = {"mean outbreak": sizes.mean(), "median": np.median(sizes), "share > 20 people": (sizes > 20).mean(), "removed roles": "".join(sorted(roles[removed])) if removed else ""}
pd.DataFrame(results).T

Removing five people at random barely helps (27 to 23). Removing the five with the most contact time halves the mean outbreak and makes the median outbreak fizzle at one person. Betweenness helps too, but less, and the reason is instructive: its list spends a slot on the administrative bridge from section 4, who has many contacts but few minutes, and *this* simulation transmits per minute of contact. The targeting metric has to match the transmission mechanism. For an airborne pathogen where any contact counts, degree or betweenness would be the right list; for one that needs prolonged contact, minutes. On a network with distinct bridges (two wards joined by one person), betweenness would pull ahead regardless.

Treat the numbers as a comparison of strategies, not a forecast. The transmission probability is invented, there is no recovery, and the static graph lets infection use a Monday contact to reach a Thursday one, which section 8 is about.

## 8. Time, and what aggregation throws away

The aggregate graph says A touched B and B touched C, so infection can go A to C. The timestamps say whether B met C *after* meeting A. If not, the path doesn't exist. Temporal networks respect that ordering; static ones don't, and overstate reachability. One look at the rhythm of the ward shows how much time structure there is.

In [ ]:
hourly = contacts.assign(hour=(contacts["t"] // 3600)).groupby("hour").size()
fig, ax = plt.subplots(figsize=(12, 3.5))
hourly.plot(ax=ax, drawstyle="steps-mid")
for d in range(1, 5): ax.axvline(d * 24, color="gray", linestyle=":", linewidth=1)
ax.set_xlabel("hours since start (Monday 1pm)"); ax.set_ylabel("contacts per hour")
ax.set_title("Contact volume by hour: shifts and nights are visible; the static graph erases them")
ax.grid(alpha=0.3); plt.show()

# How much does time ordering matter? Compare reachability on the static graph vs respecting time order (one-hour windows)
def temporal_reach(start: int) -> int:
    reached = {start}
    for _, win in contacts.assign(w=contacts["t"] // 3600).groupby("w"):
        for a, b in win[["i", "j"]].itertuples(index=False):
            if a in reached: reached.add(b)
            if b in reached: reached.add(a)
    return len(reached)

patients = [n for n in G if G.nodes[n]["role"] == "PAT"]
static = np.mean([len(nx.node_connected_component(G, p)) for p in patients])
temporal = np.mean([temporal_reach(p) for p in patients])
print(f"mean number of people reachable from a patient:  static graph {static:.0f}   respecting time order {temporal:.0f}")

The static graph says every patient can reach everyone; respecting time order, the average patient can still reach most of the ward within four days, because nurses re-contact everyone daily. The gap is small here and would be large in a sparser or shorter dataset. Know which one you computed.

## 9. Pitfalls

- **Degree isn't importance.** Pick the centrality that matches the question (reach, exposure, bridging, influence), and report which.
- **Weights.** Betweenness on a weighted graph treats weight as *distance* (higher = farther) unless you invert it. For contact time, higher should mean closer; use the unweighted version or `1/minutes`.
- **Layouts lie.** Force-directed positions are arbitrary; two nodes drawn near each other may be unconnected. Never read distance off a network picture.
- **Communities are a partition, not a truth.** Louvain is stochastic; run it several times, report modularity, and compare to something held back.
- **Static aggregation.** Reachability and path lengths on an aggregated temporal network are upper bounds.
- **Sampling.** This ward had 92% participation. The missing 8% are missing edges, and missing edges from hubs change everything downstream.

## Looking ahead
- **Reading**: Easley & Kleinberg chapters 3 (strong and weak ties) and 19 (cascading behavior). For next chapter: Jurafsky & Martin, *Speech and Language Processing*, chapter 6 (vector semantics and embeddings), free online. The rows become documents.